<a href="https://colab.research.google.com/github/ibrahimbarghout/robust-ecg-domain-generalization/blob/main/notebooks/05_patient_independent_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# PTB-XL Research Project
## 5. Patient-Independent Split

#This notebook constructs the training, validation, and test sets
#using the patient-independent stratified folds provided by PTB-XL.

#The goal is to prevent patient-level data leakage while maintaining
#similar diagnostic distributions across the three datasets.

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd
import numpy as np

In [6]:
PROJECT_PATH = "/content/drive/MyDrive/PTB-XL Research Project"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

RESULTS_PATH = os.path.join(
    PROJECT_PATH,
    "results"
)

os.makedirs(RESULTS_PATH, exist_ok=True)

print("Dataset path:")
print(DATA_PATH)

print("\nResults path:")
print(RESULTS_PATH)

Dataset path:
/content/drive/MyDrive/PTB-XL Research Project/data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3

Results path:
/content/drive/MyDrive/PTB-XL Research Project/results


In [7]:
df = pd.read_csv(
    os.path.join(DATA_PATH, "ptbxl_database.csv"),
    index_col="ecg_id"
)

print("ECG records:", len(df))
print("Unique patients:", df["patient_id"].nunique())

ECG records: 21799
Unique patients: 18869


In [9]:
TARGETS_PATH = os.path.join(
    PROJECT_PATH,
    "results",
    "ptbxl_diagnostic_targets.csv"
)

targets = pd.read_csv(
    TARGETS_PATH,
    index_col="ecg_id"
)

print("Target file:")
print(TARGETS_PATH)

print("\nTargets shape:", targets.shape)

print("\nTarget columns:")
print(targets.columns.tolist())

print("\nFirst 5 target rows:")
display(targets.head())

Target file:
/content/drive/MyDrive/PTB-XL Research Project/results/ptbxl_diagnostic_targets.csv

Targets shape: (21799, 7)

Target columns:
['NORM', 'MI', 'STTC', 'CD', 'HYP', 'ABNORMAL', 'REFERENCE_NORMAL']

First 5 target rows:


,NORM,MI,STTC,CD,HYP,ABNORMAL,REFERENCE_NORMAL
ecg_id,,,,,,,
1,1,0,0,0,0,0,1
2,1,0,0,0,0,0,1
3,1,0,0,0,0,0,1
4,1,0,0,0,0,0,1
5,1,0,0,0,0,0,1


In [10]:
print("ECG IDs in metadata:", len(df.index))
print("ECG IDs in targets:", len(targets.index))

print("\nSame ECG IDs:", df.index.equals(targets.index))

ECG IDs in metadata: 21799
ECG IDs in targets: 21799

Same ECG IDs: True


In [11]:
train_df = df[df["strat_fold"].between(1, 8)].copy()

val_df = df[df["strat_fold"] == 9].copy()

test_df = df[df["strat_fold"] == 10].copy()

print("Training ECGs:", len(train_df))
print("Validation ECGs:", len(val_df))
print("Test ECGs:", len(test_df))

print("\nTotal:", len(train_df) + len(val_df) + len(test_df))

Training ECGs: 17418
Validation ECGs: 2183
Test ECGs: 2198

Total: 21799


In [12]:
print("Unique patients:")

print(
    "Training   :", train_df["patient_id"].nunique()
)

print(
    "Validation :", val_df["patient_id"].nunique()
)

print(
    "Test       :", test_df["patient_id"].nunique()
)

Unique patients:
Training   : 15023
Validation : 1942
Test       : 1904


In [13]:
train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])
test_patients = set(test_df["patient_id"])

train_val_overlap = train_patients & val_patients
train_test_overlap = train_patients & test_patients
val_test_overlap = val_patients & test_patients

print("Train ∩ Validation:", len(train_val_overlap))
print("Train ∩ Test:", len(train_test_overlap))
print("Validation ∩ Test:", len(val_test_overlap))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [14]:
TARGET_COLUMNS = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP",
    "ABNORMAL",
    "REFERENCE_NORMAL"
]

train = train_df.join(targets[TARGET_COLUMNS])
val = val_df.join(targets[TARGET_COLUMNS])
test = test_df.join(targets[TARGET_COLUMNS])

print("Training shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

Training shape: (17418, 34)
Validation shape: (2183, 34)
Test shape: (2198, 34)


In [15]:
print("Missing target values:")

print("\nTraining:")
print(train[TARGET_COLUMNS].isna().sum())

print("\nValidation:")
print(val[TARGET_COLUMNS].isna().sum())

print("\nTest:")
print(test[TARGET_COLUMNS].isna().sum())

Missing target values:

Training:
NORM                0
MI                  0
STTC                0
CD                  0
HYP                 0
ABNORMAL            0
REFERENCE_NORMAL    0
dtype: int64

Validation:
NORM                0
MI                  0
STTC                0
CD                  0
HYP                 0
ABNORMAL            0
REFERENCE_NORMAL    0
dtype: int64

Test:
NORM                0
MI                  0
STTC                0
CD                  0
HYP                 0
ABNORMAL            0
REFERENCE_NORMAL    0
dtype: int64


In [16]:
DIAGNOSTIC_CLASSES = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP"
]

for split_name, split_data in [
    ("Training", train),
    ("Validation", val),
    ("Test", test)
]:

    print(f"\n===== {split_name} =====")

    for column in DIAGNOSTIC_CLASSES:
        count = split_data[column].sum()
        percentage = count / len(split_data) * 100

        print(
            f"{column:5s}: "
            f"{count:5d} "
            f"({percentage:5.1f}%)"
        )


===== Training =====
NORM :  7596 ( 43.6%)
MI   :  4379 ( 25.1%)
STTC :  4186 ( 24.0%)
CD   :  3907 ( 22.4%)
HYP  :  2119 ( 12.2%)

===== Validation =====
NORM :   955 ( 43.7%)
MI   :   540 ( 24.7%)
STTC :   528 ( 24.2%)
CD   :   495 ( 22.7%)
HYP  :   268 ( 12.3%)

===== Test =====
NORM :   963 ( 43.8%)
MI   :   550 ( 25.0%)
STTC :   521 ( 23.7%)
CD   :   496 ( 22.6%)
HYP  :   262 ( 11.9%)


In [17]:
for split_name, split_data in [
    ("Training", train),
    ("Validation", val),
    ("Test", test)
]:

    count = split_data["ABNORMAL"].sum()
    percentage = count / len(split_data) * 100

    print(
        f"{split_name:12s}: "
        f"{count:5d} abnormal ECGs "
        f"({percentage:5.1f}%)"
    )

Training    :  9841 abnormal ECGs ( 56.5%)
Validation  :  1232 abnormal ECGs ( 56.4%)
Test        :  1246 abnormal ECGs ( 56.7%)


In [18]:
for split_name, split_data in [
    ("Training", train),
    ("Validation", val),
    ("Test", test)
]:

    count = split_data["REFERENCE_NORMAL"].sum()
    percentage = count / len(split_data) * 100

    print(
        f"{split_name:12s}: "
        f"{count:5d} reference-normal ECGs "
        f"({percentage:5.1f}%)"
    )

Training    :  7577 reference-normal ECGs ( 43.5%)
Validation  :   951 reference-normal ECGs ( 43.6%)
Test        :   952 reference-normal ECGs ( 43.3%)


In [19]:
split_summary = pd.DataFrame({
    "Split": ["Training", "Validation", "Test"],
    "ECGs": [
        len(train),
        len(val),
        len(test)
    ],
    "Unique patients": [
        train["patient_id"].nunique(),
        val["patient_id"].nunique(),
        test["patient_id"].nunique()
    ],
    "Abnormal ECGs": [
        train["ABNORMAL"].sum(),
        val["ABNORMAL"].sum(),
        test["ABNORMAL"].sum()
    ],
    "Abnormal %": [
        train["ABNORMAL"].mean() * 100,
        val["ABNORMAL"].mean() * 100,
        test["ABNORMAL"].mean() * 100
    ]
})

display(split_summary)

,Split,ECGs,Unique patients,Abnormal ECGs,Abnormal %
0,Training,17418,15023,9841,56.499024
1,Validation,2183,1942,1232,56.436097
2,Test,2198,1904,1246,56.687898


In [20]:
SPLIT_PATH = os.path.join(
    RESULTS_PATH,
    "ptbxl_patient_independent_split.csv"
)

split_assignments = pd.DataFrame({
    "ecg_id": df.index,
    "patient_id": df["patient_id"],
    "strat_fold": df["strat_fold"],
    "split": np.select(
        [
            df["strat_fold"].between(1, 8),
            df["strat_fold"] == 9,
            df["strat_fold"] == 10
        ],
        [
            "train",
            "validation",
            "test"
        ],
        default="unknown"
    )
})

split_assignments.to_csv(
    SPLIT_PATH,
    index=False
)

print("Split saved to:")
print(SPLIT_PATH)

print("\nFile exists:", os.path.isfile(SPLIT_PATH))

Split saved to:
/content/drive/MyDrive/PTB-XL Research Project/results/ptbxl_patient_independent_split.csv

File exists: True


In [21]:
print("===== FINAL SPLIT VALIDATION =====")

print("Total ECGs:", len(split_assignments))

print("\nSplit counts:")
print(split_assignments["split"].value_counts())

print("\nPatients appearing in multiple splits:")

patient_split_counts = (
    split_assignments
    .groupby("patient_id")["split"]
    .nunique()
)

print(
    (patient_split_counts > 1).sum()
)

===== FINAL SPLIT VALIDATION =====
Total ECGs: 21799

Split counts:
split
train         17418
test           2198
validation     2183
Name: count, dtype: int64

Patients appearing in multiple splits:
0
